# mem0 in Local Mode [Step 02.05]

> **MLCourse - Agentic AI - Agent Patterns**

You have now built, by hand, the pieces of an agent memory system: extraction, a
fact store, retrieval, and a retention policy. `mem0` is a library that packages
those pieces.

This notebook runs it **entirely locally**:

```
   mem0 (local mode)
     |
     +-- LLM        -> Groq (qwen/qwen3.8-27b)      the only network call
     +-- embedder   -> fastembed, ONNX, on-CPU      local
     +-- vectors    -> FAISS, a file on disk        local
     +-- storage    -> SQLite, a file on disk       local
```

No hosted mem0 platform, no `MEM0_API_KEY`, no account. The only credential in
play is the `GROQ_API_KEY` you already have.

### What you'll learn

- Configuring mem0 with a local vector store and a local embedder.
- `add` / `search` / `get_all` / `delete` - the whole surface you need.
- The difference between mem0's **inferred** memory and **raw** memory, and why
  we use raw here.
- Where mem0 saves you work, and where it is doing exactly what you already built.

### Why it matters

Memory libraries look like magic until you have built one. Having built one, you
can read mem0's behaviour precisely: it is extraction plus a vector store plus a
scoping key. Knowing that is what lets you debug it when it forgets something.

### Prerequisites

- [04_preserve_vs_discard](04_preserve_vs_discard.ipynb) - the fact store you are about to see productised.
- [03_rag_advanced/01_embeddings_and_vectorstores](../../../03_rag_advanced) - vector search.

### Setup: environment, model, token counting, rate-limit-aware call helper


In [ ]:
import os                              # environment variables
import time                            # timing and pacing
import json                            # pretty-printing structured context
from pathlib import Path               # locating the track root
from dotenv import load_dotenv         # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we hit the repo root, then load the
# (gitignored) .env that lives inside 03_agentic_ai. Note the extra path
# segment: the walk-up lands on the REPO ROOT, not on the track folder.
TRACK = Path.cwd()
while not (TRACK / "03_agentic_ai").exists() and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / "03_agentic_ai" / ".env")

GROQ_MODEL = "qwen/qwen3.8-27b"        # hosted, fast, generous free tier
# Local alternative (documented, not used here): Ollama `llama3.1:8b` via
# `from langchain_ollama import ChatOllama`. OpenAI is never used in this course.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 300, **kw):
    """One place that constructs the chat model, so every notebook is identical."""
    return ChatGroq(model=GROQ_MODEL, temperature=temperature,
                    max_tokens=max_tokens, **kw)


# --- Token counting -----------------------------------------------------------
# Two different numbers, and it matters which one you are looking at:
#   * approx_tokens(): a LOCAL estimate using tiktoken's cl100k_base. It is not
#     the model's own tokenizer, so treat it as "within ~10%", good for
#     budgeting BEFORE you send a request.
#   * usage_metadata on the response: the provider's EXACT count. Ground truth,
#     but only available AFTER you have already paid for the call.
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def approx_tokens(text) -> int:
    """Approximate token count for a string (or anything str()-able)."""
    return len(_ENC.encode(str(text)))


# --- Rate-limit-aware calling --------------------------------------------------
# The Groq free tier allows 8000 tokens per minute. Several notebooks here make
# many small calls in a loop, so we self-pace well under the ceiling and retry
# with exponential backoff if we are throttled anyway.

TPM_BUDGET = 3500                       # deliberately conservative
_WINDOW = []                            # [(timestamp, tokens), ...]
USAGE = {"calls": 0, "in": 0, "out": 0, "seconds": 0.0}


def _pace(cost: int):
    """Sleep just enough that our rolling 60s token usage stays under budget."""
    now = time.time()
    while True:
        recent = [(t, n) for (t, n) in _WINDOW if now - t < 60]
        _WINDOW[:] = recent
        if sum(n for _, n in recent) + cost <= TPM_BUDGET or not recent:
            return
        time.sleep(min(5.0, 60 - (now - recent[0][0]) + 0.5))
        now = time.time()


def chat(messages, llm=None, temperature=0.0, max_tokens=300, retries=5):
    """Send `messages`, return the AIMessage. Paces, retries, and meters usage.

    `messages` is a list of (role, content) tuples or LangChain message objects.
    """
    llm = llm or make_llm(temperature=temperature, max_tokens=max_tokens)
    est = approx_tokens(messages) + max_tokens
    delay = 4.0
    for attempt in range(retries):
        _pace(est)
        t0 = time.time()
        try:
            out = llm.invoke(messages)
        except Exception as exc:
            if "rate_limit" in str(exc) or "429" in str(exc):
                time.sleep(delay)
                delay = min(delay * 2, 45)
                continue
            raise
        u = out.usage_metadata or {}
        _WINDOW.append((time.time(), u.get("total_tokens", est)))
        USAGE["calls"] += 1
        USAGE["in"] += u.get("input_tokens", 0)
        USAGE["out"] += u.get("output_tokens", 0)
        USAGE["seconds"] += time.time() - t0
        return out
    raise RuntimeError("still rate limited after %d attempts" % retries)


def ask(prompt: str, system: str = None, **kw) -> str:
    """Convenience wrapper: one user turn in, plain text out."""
    msgs = ([("system", system)] if system else []) + [("user", prompt)]
    return chat(msgs, **kw).content.strip()


print("model:", GROQ_MODEL)
print("key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("tokenizer:", "cl100k_base (approximation)")


### 1. Configuration

Every provider is chosen explicitly. Three things to notice:

- `vector_store: faiss` writes a file to disk. Chroma is mem0's default and works
  too; we use FAISS because it has no telemetry and no server.
- `embedder: fastembed` runs an ONNX model on the CPU. It downloads the model
  once, then needs no network at all.
- `llm: groq` is the only component that leaves your machine.

In [2]:
# Telemetry off BEFORE any vector-store import, so the notebook output stays clean.
os.environ["ANONYMIZED_TELEMETRY"] = "False"

import shutil
from mem0 import Memory

STORE_DIR = Path.cwd() / ".mem0_store"          # gitignored; local to this module
shutil.rmtree(STORE_DIR, ignore_errors=True)    # fresh start every run

CONFIG = {
    "llm": {"provider": "groq",
            "config": {"model": GROQ_MODEL, "temperature": 0.0, "max_tokens": 300}},
    "embedder": {"provider": "fastembed",
                 "config": {"model": "BAAI/bge-small-en-v1.5", "embedding_dims": 384}},
    "vector_store": {"provider": "faiss",
                     "config": {"collection_name": "spannerbox_memories",
                                "path": str(STORE_DIR),
                                "embedding_model_dims": 384}},
}

memory = Memory.from_config(CONFIG)
print("mem0 ready. store:", STORE_DIR)

The 'faiss' vector store does not support keyword search. Hybrid (BM25) scoring will be disabled and search will use semantic similarity only. To enable hybrid search, switch to a store with keyword_search support (e.g. qdrant, elasticsearch, pgvector).


[PostHog] Multiple active PostHog clients detected for the same project API key and host. Reuse one Posthog instance per app or process when possible to avoid competing background queues and missed shutdown flushes. Multiple clients are supported when intentional.


mem0 ready. store: D:\projects\python\MLCourse\03_agentic_ai\06_agent_patterns\02_memory_at_scale\.mem0_store


### 2. Two ways to add a memory

`memory.add()` has an `infer` flag, and the choice matters:

| `infer=True` (default) | `infer=False` |
|---|---|
| mem0 sends the text to the LLM with its own extraction prompt, decides what the durable facts are, and stores those | Stores exactly the text you gave it |
| One LLM call per add | Zero LLM calls |
| Convenient | You control what gets stored |

**We use `infer=False` in this notebook**, for two reasons. First, it makes the
mechanism visible: you can see precisely what went into the store. Second, a
practical one worth knowing about - mem0's built-in extraction prompt is very
large (over 8,000 tokens in the version installed here), which alone exceeds the
Groq free tier's per-minute allowance. That is a real constraint you will hit
with small quotas, and the answer is the same either way: **do your own
extraction with your own small prompt**, exactly as you did in notebook 04, and
hand mem0 the finished facts.

In [3]:
CONVERSATION_EXCERPT = """
USER: Important constraint: it must run entirely inside the EU. Our customers are
German bike shops and they will not accept US data hosting.
ASSISTANT: Understood - EU-only hosting is a hard requirement.
USER: The product is called Spannerbox, and our budget is 12,000 EUR for six months.
ASSISTANT: Noted.
USER: We've decided on Postgres and Django. That's locked in.
ASSISTANT: Good - Django's admin will save you weeks.
USER: Our first pilot customer is Radhaus Krueger in Freiburg, starting in March.
ASSISTANT: A named pilot with a date is the most useful thing you have.
"""

EXTRACT = """From the conversation below, extract durable facts about the user's project.

One fact per line, no numbering, no preamble. Include every name, number and date
verbatim. Skip small talk and anything that was not decided.

CONVERSATION:
""" + CONVERSATION_EXCERPT

facts = [ln.strip().lstrip("-* ").strip()
         for ln in chat([("user", EXTRACT)], temperature=0.0,
                        max_tokens=220).content.strip().splitlines()
         if ln.strip()]

for f in facts:
    print("-", f)

- The product must run entirely inside the EU.
- The customers are German bike shops.
- US data hosting is not acceptable.
- The product is called Spannerbox.
- The budget is 12,000 EUR for six months.
- The database is Postgres.
- The framework is Django.
- The first pilot customer is Radhaus Krueger in Freiburg.
- The pilot starts in March.


In [4]:
USER_ID = "founder_01"

for f in facts:
    result = memory.add(f, user_id=USER_ID, infer=False)
    print("stored:", result["results"][0]["memory"][:64])

D:\projects\python\MLCourse\.venv\Lib\site-packages\typer\__init__.py:24: DeprecationWarning: 'click.utils.get_binary_stream' is deprecated and will be removed in Click 9.0.
  from click.utils import get_binary_stream as get_binary_stream
D:\projects\python\MLCourse\.venv\Lib\site-packages\typer\__init__.py:25: DeprecationWarning: 'click.utils.get_text_stream' is deprecated and will be removed in Click 9.0.
  from click.utils import get_text_stream as get_text_stream


stored: The product must run entirely inside the EU.
stored: The customers are German bike shops.
stored: US data hosting is not acceptable.
stored: The product is called Spannerbox.


stored: The budget is 12,000 EUR for six months.
stored: The database is Postgres.


stored: The framework is Django.
stored: The first pilot customer is Radhaus Krueger in Freiburg.
stored: The pilot starts in March.


> **`user_id` is the scoping key and it is not optional in practice.** Every
> memory is filed under it, and every search filters by it. Forgetting to scope -
> or reusing one id across customers - is how a memory system leaks one user's
> facts into another user's conversation. mem0 also supports `agent_id` and
> `run_id` for scoping by agent and by session.

### 3. Searching

`search` is a vector similarity query over that user's memories. It is a local
embedding lookup - no LLM call, so it is fast and free.

Note the API detail: scoping goes in `filters=`, not as a top-level argument.

In [5]:
def show_search(q, limit=3):
    hits = memory.search(q, filters={"user_id": USER_ID}, limit=limit)["results"]
    print("Q: %s" % q)
    for h in hits:
        print("   %.3f  %s" % (h["score"], h["memory"]))
    print()
    return hits


show_search("where are we allowed to host the data?")
show_search("which web framework are we using?")
show_search("who is our first customer?")

Q: where are we allowed to host the data?
   0.714  US data hosting is not acceptable.
   0.593  The database is Postgres.
   0.551  The framework is Django.
   0.527  The product must run entirely inside the EU.
   0.493  The product is called Spannerbox.
   0.469  The budget is 12,000 EUR for six months.
   0.457  The pilot starts in March.
   0.444  The first pilot customer is Radhaus Krueger in Freiburg.
   0.441  The customers are German bike shops.

Q: which web framework are we using?
   0.654  The framework is Django.
   0.548  The database is Postgres.
   0.511  US data hosting is not acceptable.
   0.509  The product is called Spannerbox.
   0.463  The product must run entirely inside the EU.
   0.454  The budget is 12,000 EUR for six months.
   0.454  The pilot starts in March.
   0.443  The customers are German bike shops.
   0.439  The first pilot customer is Radhaus Krueger in Freiburg.



Q: who is our first customer?
   0.575  The first pilot customer is Radhaus Krueger in Freiburg.
   0.543  The customers are German bike shops.
   0.531  The database is Postgres.
   0.515  The pilot starts in March.
   0.510  The product is called Spannerbox.
   0.500  The framework is Django.
   0.475  The product must run entirely inside the EU.
   0.470  US data hosting is not acceptable.
   0.449  The budget is 12,000 EUR for six months.



[{'id': '5054f615-6a53-422f-a01d-a34b1b0b3010',
  'memory': 'The first pilot customer is Radhaus Krueger in Freiburg.',
  'hash': 'f8d702d741bdef469d9120687242aaf0',
  'metadata': None,
  'score': 0.5747572352549364,
  'created_at': '2026-08-31T01:40:40.146333+00:00',
  'updated_at': '2026-08-31T01:40:40.146333+00:00',
  'user_id': 'founder_01',
  'role': 'user'},
 {'id': '93c2fd22-efe3-4283-9406-cfaf3442e956',
  'memory': 'The customers are German bike shops.',
  'hash': '3608bdbd375c36a7e852b2a944dbb552',
  'metadata': None,
  'score': 0.5433193212705496,
  'created_at': '2026-08-31T01:40:39.617745+00:00',
  'updated_at': '2026-08-31T01:40:39.617745+00:00',
  'user_id': 'founder_01',
  'role': 'user'},
 {'id': '96aceb08-c784-4428-ad1d-3934e56fecfd',
  'memory': 'The database is Postgres.',
  'hash': 'fd587b132201ebd933317758be8b10c3',
  'metadata': None,
  'score': 0.5308801713476994,
  'created_at': '2026-08-31T01:40:39.917457+00:00',
  'updated_at': '2026-08-31T01:40:39.917457+00:0

Notice the third query: it finds the pilot-customer fact even though the stored
text says "pilot customer" and the query says "first customer". That is the
embedder doing its job, and it is the concrete advantage of a vector store over
the flat fact list from notebook 04 - **you can ask in your own words.**

And notice what it does *not* do: it returns the top-k by similarity regardless
of whether they are relevant. A low score is still returned. Score thresholding
is your job.

### Retrieval-augmented answering: search, then answer using only what came back.


In [ ]:
def answer_with_memory(question, limit=3, threshold=0.30):
    hits = memory.search(question, filters={"user_id": USER_ID}, limit=limit)["results"]
    kept = [h for h in hits if h["score"] >= threshold]
    notes = "\n".join("- " + h["memory"] for h in kept) or "(no relevant memories)"
    out = chat([("system", "Answer using ONLY these remembered facts. If they do "
                           "not cover it, say UNKNOWN. One sentence."),
                ("user", "Remembered facts:\n%s\n\nQuestion: %s" % (notes, question))],
               temperature=0.0, max_tokens=70)
    return out.content.strip(), len(hits), len(kept)


for q in ["What is the monthly hosting constraint?",
          "What is our budget?",
          "What is the CEO's favourite colour?"]:
    a, n, k = answer_with_memory(q)
    print("Q: %s\n   retrieved %d, kept %d above threshold\n   A: %s\n" % (q, n, k, a))


The third question is the important one. There is no memory of a favourite
colour, the retriever still returns its three nearest neighbours, and the
score threshold is what stops those irrelevant memories from being presented to
the model as if they were the answer.

> **Pitfall.** Without a threshold, a vector store answers *every* question with
> *something*. That "something" then appears in the context labelled as a
> remembered fact, and the model will use it. This is the single most common
> memory bug in production RAG-backed agents.

### 4. Inspecting and editing the store

Memory you cannot inspect is memory you cannot debug. mem0 gives you the whole
store, and per-memory delete.

In [7]:
all_mem = memory.get_all(filters={"user_id": USER_ID})["results"]
print("%d memories for %s\n" % (len(all_mem), USER_ID))
for m in all_mem:
    print("  %s  %s" % (m["id"][:8], m["memory"][:70]))

9 memories for founder_01

  e999c118  The product must run entirely inside the EU.
  93c2fd22  The customers are German bike shops.
  80fbca7d  US data hosting is not acceptable.
  d270f0c1  The product is called Spannerbox.
  54b59bb6  The budget is 12,000 EUR for six months.
  96aceb08  The database is Postgres.
  310b4e7e  The framework is Django.
  5054f615  The first pilot customer is Radhaus Krueger in Freiburg.
  421286df  The pilot starts in March.


### Supersession, mem0-style: add the new fact, delete the stale one.


In [ ]:
old = [m for m in all_mem if "12,000" in m["memory"] or "12000" in m["memory"]]
memory.add("The budget was raised to 18,000 EUR for the first six months.",
           user_id=USER_ID, infer=False)
for m in old:
    memory.delete(memory_id=m["id"])
    print("deleted stale:", m["memory"][:60])

a, _, _ = answer_with_memory("What is our budget?")
print("\nafter update ->", a)


Compare this with notebook 04's append-only store, which *marked* the old fact
stale rather than deleting it. mem0's `delete` is a real delete: the audit trail
is gone. If you need history - and for anything financial, medical or contractual
you do - keep your own append-only log and treat the vector store as a
**derived index** you can rebuild, not as the source of truth.

### 5. What mem0 gives you, and what it does not

| mem0 handles | You still own |
|---|---|
| Storage, embedding, similarity search | **What counts as a durable fact** |
| Scoping by user / agent / session | The **score threshold** |
| Metadata, timestamps, ids | **Supersession policy** and audit history |
| A stable API across vector stores | Deciding when to write at all |

The column on the right is the entire content of notebook 04. Libraries move the
plumbing; the policy is still yours, and the policy is where memory systems
actually fail.

In [9]:
print("Files mem0 wrote (all local, all yours):")
for p in sorted(STORE_DIR.rglob("*")):
    if p.is_file():
        print("  %-42s %8d bytes" % (p.relative_to(STORE_DIR), p.stat().st_size))
print()
print("Add %s to .gitignore." % STORE_DIR.name)

Files mem0 wrote (all local, all yours):
  spannerbox_memories.faiss                     13869 bytes
  spannerbox_memories.json                       3963 bytes
  spannerbox_memories_entities.faiss               45 bytes
  spannerbox_memories_entities.json                44 bytes

Add .mem0_store to .gitignore.


### 6. Pitfalls

- **Forgetting `user_id`.** Unscoped memories leak between users. Scope every
  add and every search.
- **No score threshold.** The store answers everything, relevantly or not.
- **Treating the vector store as the source of truth.** It is a derived index.
  Keep the log.
- **Relying on `infer=True` blindly.** It is one LLM call per add, with a prompt
  you did not write and cannot see in your logs. Extract your own facts when the
  facts matter.
- **Assuming local means free.** The embedder is free; the extraction call is not.

### Recap

| Idea | Takeaway |
|---|---|
| Fully local | FAISS + fastembed on disk; Groq is the only network call |
| `infer=False` | You choose what is stored, with your own small prompt |
| Scope everything | `user_id` on every add and every search |
| Threshold the scores | Or the store will confidently answer questions it cannot |
| Index, not truth | Keep an append-only log you can rebuild from |

### Module complete

You can now diagnose an overgrown conversation, compress it with a rolling
summary, structure it as a hierarchy with drill-down, write a retention policy
that survives compression, and run a real memory library entirely on your own
machine.

**Next module:** [03_sampling_and_search](../03_sampling_and_search) - spending
*more* calls, deliberately, to buy accuracy - and measuring whether it worked.